# Orca Core (Novus) — Combined Safety + Calibration SFT (Kaggle, joint pass on top of v2, plain PEFT)

**Why this notebook exists**: two sequential fine-tuning rounds on `orca-core` — first probe-grounded safety DPO (jailbreak block rate 0%→20% strict), then a separate calibration-fix SFT round on top of it (calibration 16.7%→33.3%) — produced a real regression: the calibration SFT round, trained without replaying any safety-refusal examples, caused catastrophic forgetting of the DPO round's jailbreak gains (block rate collapsed back to 0% strict, `orca-core-calibration`'s measured safety score 37.5/100 vs `orca-core-dpo`'s 47.0/100 — see `redteam_orca-core-calibration.json`).

**The fix**: train BOTH skills in a single joint SFT pass instead of two sequential ones, starting fresh from the v2 SFT adapter (the pre-DPO foundation) rather than continuing from either downstream checkpoint. Combined dataset: the 3 real probe-grounded safety-refusal examples (`core_probe_grounded_safety_dpo_20260724.jsonl`'s `prompt`+`chosen` pairs, reformatted as plain SFT demonstrations — DPO's `rejected` side isn't needed here since this is demonstration learning, not preference learning) + the 60 premise-correction calibration examples, shuffled together into one 56-train/7-eval split. A single joint pass over both skills together means neither skill's data gets a chance to overwrite the other's weights afterward.

**Why plain PEFT, not Unsloth, and plain `transformers.Trainer`, not trl's `SFTTrainer`**: exact same infra already proven working on `orca_core_calibration_sft_kaggle_v1.ipynb` after 11 failed attempts diagnosing real Unsloth/xformers/trl bugs (Unsloth's patched attention has no working xformers backend for Llama-3.1's grouped-query-attention on T4; trl's `SFTTrainer` has a real bug — huggingface/trl#6483 — when `device_map="auto"` + 4-bit loading wraps `.forward` via accelerate hooks). Reusing that exact working pipeline rather than re-deriving it.

**Continuing from v2, not from DPO or calibration**: loads the base model + attaches the `orca-core-sft-adapter-v2` LoRA (the original, pre-DPO SFT checkpoint — same base every downstream round has started from) as a trainable PEFT model.

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "0"

# Plain transformers/peft/bitsandbytes stack -- no unsloth, no xformers, no trl.
# See orca_core_calibration_sft_kaggle_v1.ipynb's own notes for why each of
# these is avoided -- same proven pipeline reused here unchanged.
#
# transformers pinned <5 -- unpinned installs pull transformers 5.x, whose new
# internal weight-conversion registry breaks downstream merge tooling.
#
# torchao pinned >=0.16.0 -- peft's LoRA-merge dispatch checks torchao's
# version and raises ImportError on Kaggle's older preinstalled torchao
# during the merge step.
!pip install -q "transformers<5" "torchao>=0.16.0" peft bitsandbytes accelerate datasets

## Find the combined dataset and the v2 SFT adapter

In [ ]:
import glob, os

train_matches = glob.glob('/kaggle/input/**/orca_core_combined_train_v2.jsonl', recursive=True)
eval_matches  = glob.glob('/kaggle/input/**/orca_core_combined_eval_v2.jsonl', recursive=True)
adapter_matches = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)

print('Train file:', train_matches)
print('Eval file:', eval_matches)
print('Adapter config:', adapter_matches)

if not train_matches or not adapter_matches:
    raise FileNotFoundError(
        "Need both the combined train/eval dataset attached (dataset_sources: "
        "orca-core-combined-safety-calibration-v1) and the v2 SFT adapter "
        "attached (dataset_sources: orca-core-sft-adapter-v2) — check the "
        "notebook's kernel-metadata.json."
    )

train_path = train_matches[0]
eval_path = eval_matches[0] if eval_matches else None
adapter_dir = os.path.dirname(adapter_matches[0])
print('Using adapter_dir:', adapter_dir)

In [ ]:
import json

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
    return records

raw_train = load_jsonl(train_path)
raw_eval  = load_jsonl(eval_path) if eval_path else raw_train[:max(1, len(raw_train)//10)]
print(f'train={len(raw_train)} eval={len(raw_eval)}')

## Load base model (4-bit, plain bitsandbytes) + attach the v2 SFT adapter (continuing from the shared foundation, not from a downstream checkpoint)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

max_seq_length = 2048
base_model_name = "unsloth/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)
# Load the tokenizer from the BASE model, not adapter_dir -- the v2 SFT
# adapter's saved tokenizer_config.json references a tokenizer class
# ("TokenizersBackend") that this transformers version doesn't recognize
# (real error hit on the first run of this notebook), likely saved by a
# different Kaggle environment snapshot back when that adapter was first
# trained. The tokenizer itself doesn't change across LoRA adapters for the
# same base model, so loading it from base_model_name is functionally
# identical and sidesteps the stale config entirely.
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = prepare_model_for_kbit_training(model)

# Attach the v2 SFT adapter, trainable — this joint pass trains BOTH the
# safety-refusal and calibration skills together on top of the shared
# foundation, instead of continuing from either downstream checkpoint (which
# is what caused the sequential-forgetting regression this notebook exists
# to fix).
model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=True)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset

def _tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=max_seq_length)

train_ds = Dataset.from_list([{"text": ex["text"]} for ex in raw_train]).map(_tokenize, remove_columns=["text"])
eval_ds  = Dataset.from_list([{"text": ex["text"]} for ex in raw_eval]).map(_tokenize, remove_columns=["text"])
print(f'train_ds={len(train_ds)} eval_ds={len(eval_ds)}')

## Train

Small combined dataset (56 train examples: 3 safety-refusal + ~53 calibration,
shuffled together) — 3 epochs, same effective-batch-16 sizing as prior runs.
Plain `transformers.Trainer`, not trl's `SFTTrainer` (see markdown intro for
why). No periodic eval during training (`eval_strategy="no"`) — a real
`OutOfMemoryError` hit inside the evaluation loop on the first run of this
notebook (training itself got past step 5 fine); eval isn't needed for the
actual goal here (the trained adapter), so removing it eliminates the OOM
trigger point directly. No mid-training checkpointing (`save_strategy="no"`)
— the adapter-save cell right after training is the real safety net.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import time

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=2,
    save_strategy="no",
    output_dir="/kaggle/working/output",
    eval_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=data_collator,
)

print("[train] starting combined safety+calibration QLoRA training (plain PEFT + plain Trainer, no unsloth, no trl)...")
t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
print(f"[train] done in {elapsed:.1f} min")

## Save the updated adapter immediately (before merge/export)

In [ ]:
adapter_out_dir = "/kaggle/working/adapter_combined"
model.save_pretrained(adapter_out_dir)
tokenizer.save_pretrained(adapter_out_dir)
print(f"[adapter] saved to {adapter_out_dir} — combined safety+calibration weights are now safe on disk.")
!ls -la {adapter_out_dir}

## Merge LoRA + export GGUF — plain PEFT, base model reloaded in fp16 (never 4-bit)

In [ ]:
import gc, torch as _torch

# Free the 4-bit training model before loading a second fp16 copy — T4 has
# 16GB VRAM, not enough for both at once.
del model, trainer
gc.collect()
_torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM as _AutoModelForCausalLM
from peft import PeftModel as _PeftModel

print("[load] loading base model in fp16 for merge...")
base_fp16 = _AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=_torch.float16,
    device_map="auto",
)

print("[merge] attaching the just-trained adapter and merging...")
merge_model = _PeftModel.from_pretrained(base_fp16, adapter_out_dir)
merged = merge_model.merge_and_unload()
print("[merge] done — merged is a plain AutoModelForCausalLM, no PEFT wrapper, no quant config")

assert getattr(merged.config, "quantization_config", None) is None, (
    "merged model still has a quantization_config — something upstream "
    "changed and this notebook's core assumption no longer holds, stop here."
)
print("[check] confirmed: no quantization_config on the merged model")

import shutil
shutil.rmtree("/tmp/merged_clean", ignore_errors=True)
merged.save_pretrained("/tmp/merged_clean", safe_serialization=True)
tokenizer.save_pretrained("/tmp/merged_clean")
print("[save] merged model saved to /tmp/merged_clean")
!ls -la /tmp/merged_clean

## Convert to GGUF using the standard llama.cpp project directly, then quantize to Q4_K_M

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
!pip install -q -r /tmp/llama.cpp/requirements.txt

In [ ]:
import os

os.makedirs("/tmp/gguf_out", exist_ok=True)
f16_path = "/tmp/gguf_out/orca-core-combined.F16.gguf"

print("[convert] running llama.cpp's convert_hf_to_gguf.py...")
!python3 /tmp/llama.cpp/convert_hf_to_gguf.py /tmp/merged_clean --outfile {f16_path} --outtype f16
print("[convert] done, checking output:")
!ls -la /tmp/gguf_out

## Build llama.cpp's quantize tool and produce the final Q4_K_M GGUF

In [ ]:
!cmake -B /tmp/llama.cpp/build -S /tmp/llama.cpp -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=OFF
!cmake --build /tmp/llama.cpp/build --config Release -j --target llama-quantize

In [ ]:
import glob

quantize_bin_candidates = glob.glob("/tmp/llama.cpp/build/**/llama-quantize", recursive=True)
print("[quantize] llama-quantize binary found at:", quantize_bin_candidates)

if not quantize_bin_candidates:
    raise RuntimeError(
        "llama-quantize binary not found after build -- check the cmake build "
        "output above for the real error before assuming this notebook's logic "
        "is wrong; this is a standard llama.cpp build step, not custom code."
    )

q4_path = "/tmp/gguf_out/orca-core-combined.Q4_K_M.gguf"
!{quantize_bin_candidates[0]} {f16_path} {q4_path} Q4_K_M
print("[quantize] done, checking output:")
!ls -la /tmp/gguf_out

## Copy the final Q4_K_M GGUF to /kaggle/working (only the final file — the F16 intermediate stays in /tmp, too big for the 19.5GB working quota)

In [ ]:
import shutil

dest = "/kaggle/working/orca-core-combined.Q4_K_M.gguf"
shutil.copy(q4_path, dest)
print(f"[export] copied final quantized model to {dest}")
print("\nNext: click 'Save Version' -> 'Save & Run All (Commit)' at the top right.")